In [0]:
#Install Vector Search client
%pip install databricks-vectorsearch
dbutils.library.restartPython()

In [0]:
%sql
ALTER TABLE dbw_agentic_ai_dev.telco_ai.customer_note_embeddings
SET TBLPROPERTIES (
  delta.enableChangeDataFeed = true
);

In [0]:
# Step 1: import Vector Search client
from databricks.vector_search.client import VectorSearchClient

vsc = VectorSearchClient()

#Step 2: Define names
VECTOR_SEARCH_ENDPOINT_NAME = "telco-vector-search-endpoint"
SOURCE_TABLE = "dbw_agentic_ai_dev.telco_ai.customer_note_embeddings"
INDEX_NAME = "dbw_agentic_ai_dev.telco_ai.customer_note_embeddings_index"

#SOURCE_TABLE = Delta table with embeddings
#INDEX_NAME = Vector Search index built from that table
#ENDPOINT = Serving layer for vector search queries

# Step 3: Create Vector Search endpoint
#vsc.create_endpoint(name=VECTOR_SEARCH_ENDPOINT_NAME, endpoint_type="STANDARD")

# Step 4: Create Delta Sync Index
#Because table already has an embedding column:

embedding_dimension = len(
    spark.table(SOURCE_TABLE)
         .select("embedding")
         .first()["embedding"]
)

index = vsc.create_delta_sync_index(
    endpoint_name=VECTOR_SEARCH_ENDPOINT_NAME,
    source_table_name=SOURCE_TABLE,
    index_name=INDEX_NAME,
    pipeline_type="TRIGGERED",
    primary_key="customer_id",
    embedding_dimension=embedding_dimension,
    embedding_vector_column="embedding",
    columns_to_sync=["customer_id", "note"]
)


In [0]:
#Step 5: Sync the index
index.sync()


In [0]:
index.describe()

In [0]:
#Step 6: Create embedding for a user question
from databricks.sdk import WorkspaceClient
w = WorkspaceClient()
question = "Which customers are likely to cancel service?"

response = w.serving_endpoints.query(
    name="databricks-gte-large-en",
    input=[question]
)

question_embedding = [float(x) for x in response.data[0].embedding]
#Same model = same vector space.

#Step 7: Query Vector Search index
results = index.similarity_search(
    query_vector=question_embedding,
    columns=["customer_id", "note"],
    num_results=3
)

results

In [0]:
#vsc.delete_index(    endpoint_name="telco-vector-search-endpoint",    index_name="dbw_agentic_ai_dev.telco_ai.customer_note_embeddings_index")

In [0]:
vsc.get_index(
    endpoint_name="telco-vector-search-endpoint",
    index_name="dbw_agentic_ai_dev.telco_ai.customer_note_embeddings_index"
)